#### header
'n', 'quantile', 'distribution', 'param1', 'param2', 'seed', 'estimated quantile', 'true quantile', 'elapsed time', 'updates/s', 'epsilon', 'estimated sensitivity', 'chunks', 'laplace dp estimate', 'DP relative error'

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import math

test_names = {'chunks': 4, 'distribution':8, 'epsilon':4, 'length':4, 'quantile':4}
dist_name = ['normal', 'cauchy', 'uniform', 'exponential', 'chisquared', 'gamma', 'lognormal', 'extremevalue']
############################################################

q_values = ["0.1", "0.3",  "0.5", "0.99"]
q_default = "0.99"

ni_values = ["10000000", "50000000", "75000000", "100000000"]
ni_default = "10000000"

d_values = ["1", "2", "3", "4", "5", "6", "7", "8"]
d_default = "1"

eps_values = ["0.1", "0.5", "1", "2"]
eps_default = "1"

k_values = ["2", "4", "8", "16"]
k_default = "4"

seed_base = 16033099
seed_step = 127
reps = 100
############################################################

In [2]:

for name,num in test_names.items():
    avgDfs = []
    for c in range(1, num+1):
        dfs = [pd.read_csv(f'raw/test_stream_{name}_{c}/test_stream_{name}_{c}_{i}.csv', names=['n', 'q', 'dist', 'param1', 'param2', 'seed', 'est_q', 'true_q', 'time', 'updates', 'epsilon', 'sensitivity', 'chunks', 'laplace_estimate', 'nae', 're']) for i in range(1,reps+1)]
        data = pd.concat(dfs, ignore_index=True)
        data.drop(columns=['re'], inplace=True)
        data['distr'] = data['dist'].map(str.strip).map(dist_name.index).astype(int)
        data.drop(columns=['dist'], inplace=True)
        data = data.iloc[:,[0,1,14,2,3,4,5,6,7,8,9,10,11,12,13]]
        data_avg = data.mean().to_frame().transpose()
        data_avg['distr'] = data_avg['distr'].astype(int)
        ci = stats.t.interval(0.95, df=reps-1, loc = data['nae'].mean(), scale = data['nae'].std()/math.sqrt(reps))
        data_avg['nae_cil'] = ci[0]
        data_avg['nae_cir'] = ci[1]
        avgDfs.append(data_avg)
    df = pd.concat(avgDfs, ignore_index=True)
    df.to_csv(f'test_on_{name}.csv', index=False)

In [5]:
name, num = 'distribution', 8
for c in range(1, num+1):
    dfs = [pd.read_csv(f'raw/test_stream_{name}_{c}/test_stream_{name}_{c}_{i}.csv', names=['n', 'q', 'dist', 'param1', 'param2', 'seed', 'est_q', 'true_q', 'time', 'updates', 'epsilon', 'sensitivity', 'chunks', 'laplace_estimate', 'nae', 're']) for i in range(1,reps+1)]
    data = pd.concat(dfs, ignore_index=True)
    #data.drop(columns=['re'], inplace=True)
    data['distr'] = data['dist'].map(str.strip).map(dist_name.index).astype(int)
    data.drop(columns=['dist'], inplace=True)
    data = data.iloc[:,[0,1,15,2,3,4,5,6,7,8,9,10,11,12,13,14]]
    data.to_csv(f'test_on_{name}_{c}.csv', index=False)